In [1]:
import pandas as pd
import geopandas as gpd
from sklearn.model_selection import RandomizedSearchCV,train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
import numpy as np

In [2]:
data = pd.read_csv("../data/final_df_processed_with_proximity.csv")

In [3]:
data.sample(10)

,elevation_m,h3_index,has_turbine,impervious,land_type,slope_deg,soil_type,protected_area,in_wdpa,pop_density,lon,lat,road_dist_km,transmission_line_dist_km,wind_speed,airport_dist_km
158057,1150.0,8926a5b54d7ffff,False,0,11,14.572644,1,0.0,1.0,0.000000,-103.353432,43.710664,42.767121,5.970802,4.579010,20.473092
221428,223.0,8926629a47bffff,True,0,13,3.071436,1,0.0,0.0,0.001857,-88.938157,40.833772,8.178710,7.828394,3.884827,5.016695
253848,400.0,8927564cb2bffff,False,0,14,1.854334,1,0.0,0.0,0.000000,-93.099123,46.266576,20.780065,11.215983,3.168655,16.550610
214912,927.0,8926f36f463ffff,False,0,13,1.483991,1,0.0,0.0,0.000000,-101.321415,36.837205,175.107095,1.298044,4.511282,8.625146
224178,136.0,892888886b7ffff,False,0,13,0.927410,1,0.0,0.0,0.000000,-118.713015,46.034114,34.212227,1.863540,2.642604,13.183951
129608,1851.0,8926882a85bffff,True,0,11,1.495639,0,0.0,0.0,0.000000,-104.488942,37.796901,29.618548,0.896566,3.271219,9.210470
118452,493.0,8928f2c353bffff,False,0,10,4.557342,1,0.0,0.0,0.000000,-120.975108,45.854390,22.354229,0.483827,1.942352,5.190601
36224,527.0,892a8386413ffff,False,0,7,17.350273,1,0.0,0.0,0.000000,-80.216120,38.900448,22.618883,1.553717,2.106071,12.307508
149335,721.0,8926c6ce9cbffff,False,0,11,3.471140,1,0.0,0.0,0.000000,-100.114328,36.810731,168.428657,22.528239,4.485240,25.017025
246846,5.0,892af2b0a6bffff,True,0,13,4.357400,1,0.0,0.0,0.000000,-76.431520,36.291696,10.479694,5.649942,2.450218,5.989190


## Add feature interactions and log transform

In [4]:
# Log important distances
data['log_road_dist_km'] = np.log(data['road_dist_km'] + 1)
data['log_transmission_line_dist_km'] = np.log(data['transmission_line_dist_km'] + 1)


In [5]:
# Feature interactions
data['log_road_dist_km_x_transmission_line_dist_km'] = data['log_road_dist_km'] * data['log_transmission_line_dist_km']
data['slope_x_elevation_m'] = data['slope_deg'] * data['elevation_m']
data['wind_speed_x_elevation_m'] = data['wind_speed'] * data['elevation_m']
data['wind_speed_x_slope_deg'] = data['wind_speed'] * data['slope_deg']

In [6]:
print(data.columns)
print(data.dtypes)

Index(['elevation_m', 'h3_index', 'has_turbine', 'impervious', 'land_type',
       'slope_deg', 'soil_type', 'protected_area', 'in_wdpa', 'pop_density',
       'lon', 'lat', 'road_dist_km', 'transmission_line_dist_km', 'wind_speed',
       'airport_dist_km', 'log_road_dist_km', 'log_transmission_line_dist_km',
       'log_road_dist_km_x_transmission_line_dist_km', 'slope_x_elevation_m',
       'wind_speed_x_elevation_m', 'wind_speed_x_slope_deg'],
      dtype='str')
elevation_m                                     float64
h3_index                                            str
has_turbine                                        bool
impervious                                        int64
land_type                                         int64
slope_deg                                       float64
soil_type                                         int64
protected_area                                  float64
in_wdpa                                         float64
pop_density              

## Train Model

In [8]:
X = data.drop(columns=['has_turbine', 'lat', 'lon', 'h3_index', 'road_dist_km', 'transmission_line_dist_km'])
y = data['has_turbine']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=47)

In [9]:
model = XGBClassifier(tree_method='hist')

param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [4, 6, 8, 10],
    'min_child_weight': [1, 3, 5, 7],
    'subsample': [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'reg_alpha': [0, 0.01, 0.1, 1.0],
    'reg_lambda': [1, 2, 5, 10],
    'scale_pos_weight': [1, len(y_train[y_train==0]) / len(y_train[y_train==1])],
}

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=100,
    scoring='f1_weighted',
    n_jobs=-1,
    cv=5,
    verbose=1,
    random_state=42
)

search.fit(X_train, y_train)

print("Best parameters:")
print(search.best_params_)
print(f"Best CV f1_weighted: {search.best_score_:.4f}")

Fitting 5 folds for each of 100 candidates, totalling 500 fits


Best parameters:
{'subsample': 1.0, 'scale_pos_weight': 1, 'reg_lambda': 10, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.2, 'colsample_bytree': 1.0}
Best CV f1_weighted: 0.9555


In [10]:
# Use the best estimator in a pipeline with scaling
best_model = search.best_estimator_

best_model.fit(X_train, y_train)

# Evaluate with cross_val_score
scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"Cross-validation accuracy scores: {scores}")
print(f"Mean cross-validation accuracy: {scores.mean():.4f}")

Cross-validation accuracy scores: [0.95465394 0.95520286 0.95489152 0.95634741 0.95489152]
Mean cross-validation accuracy: 0.9552


In [11]:
# Predict on the test set using the best pipeline
predictions = best_model.predict(X_test)

# Print classification report and confusion matrix
print("Classification report for best pipeline:")
print(classification_report(y_test, predictions))
print("Confusion matrix for best pipeline:")
print(confusion_matrix(y_test, predictions))

Classification report for best pipeline:
              precision    recall  f1-score   support

       False       0.98      0.97      0.97     39237
        True       0.90      0.94      0.92     13138

    accuracy                           0.96     52375
   macro avg       0.94      0.95      0.95     52375
weighted avg       0.96      0.96      0.96     52375

Confusion matrix for best pipeline:
[[37864  1373]
 [  740 12398]]


In [19]:
import joblib

# Save
best_model.fit(X_train, y_train)  # train on all data first
joblib.dump(best_model, 'viability_model.pkl')

['viability_model.pkl']

In [20]:
loaded_model = joblib.load('../../models/viability_model.pkl')
new_preds = loaded_model.predict(X_test)

In [21]:
# Print classification report and confusion matrix
print("Classification report for best pipeline:")
print(classification_report(y_test, new_preds))
print("Confusion matrix for best pipeline:")
print(confusion_matrix(y_test, new_preds))

Classification report for best pipeline:
              precision    recall  f1-score   support

       False       0.98      0.97      0.97     39237
        True       0.90      0.94      0.92     13138

    accuracy                           0.96     52375
   macro avg       0.94      0.95      0.95     52375
weighted avg       0.96      0.96      0.96     52375

Confusion matrix for best pipeline:
[[37864  1373]
 [  740 12398]]


In [22]:
X.head()

,elevation_m,impervious,land_type,slope_deg,soil_type,protected_area,in_wdpa,pop_density,wind_speed,airport_dist_km,log_road_dist_km,log_transmission_line_dist_km,log_road_dist_km_x_transmission_line_dist_km,slope_x_elevation_m,wind_speed_x_elevation_m,wind_speed_x_slope_deg
0,96.0,0,1,5.398381,1,0.0,0.0,0.000000,2.658464,15.655107,2.782325,0.677070,1.883829,518.244559,255.212548,14.351401
1,37.0,0,1,3.557861,1,0.0,0.0,0.000000,2.468661,6.395007,4.319651,1.147896,4.958511,131.640842,91.340472,8.783153
2,22.0,0,1,1.104470,1,0.0,1.0,0.000000,2.591707,19.674052,3.819563,1.715232,6.551436,24.298350,57.017548,2.862464
3,139.0,0,1,1.488473,1,0.0,0.0,0.341814,2.472444,6.566614,0.195413,0.666765,0.130295,206.897773,343.669647,3.680166
4,136.0,0,1,2.996490,1,0.0,0.0,0.173283,2.703351,7.993072,2.907971,0.955141,2.777523,407.522634,367.655671,8.100563


In [23]:
# Predict on known bad locations
water_mask = X_test['land_type'] == 11
urban_mask = X_test['land_type'].isin([23, 24])
protected_mask = X_test['protected_area'] == 1

print(f"Water predictions (should be ~0): {best_model.predict(X_test[water_mask]).mean():.3f}")
print(f"Urban predictions (should be ~0): {best_model.predict(X_test[urban_mask]).mean():.3f}")
print(f"Protected predictions (should be ~0): {best_model.predict(X_test[protected_mask]).mean():.3f}")

Water predictions (should be ~0): 0.345
Urban predictions (should be ~0): nan
Protected predictions (should be ~0): 0.058


/var/folders/5g/m0r4gtb92cd65m1l48wr7sf00000gn/T/ipykernel_33404/2227514855.py:7: RuntimeWarning: Mean of empty slice
  print(f"Urban predictions (should be ~0): {best_model.predict(X_test[urban_mask]).mean():.3f}")
/Users/lukepitstick/Projects/Data-Science/renewably-wind/src/python/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
